In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings("ignore", category = FutureWarning)

import importlib
import mis_funciones
importlib.reload(mis_funciones)

import gspread
from google.oauth2.service_account import Credentials
from ortools.sat.python import cp_model

In [2]:
# Configuración inicial
id_libro = "1a7vwC8QZAb7KI2elM_vJOZGSQzitnZ0Xw6BWFlG5guw"
id_hoja_operadores = "1408324300"
id_hoja_lista_actividades = "830504512"
id_hoja_lista_actividades_actualizado = "1447386758"
id_hoja_areas = "634917592"
id_hoja_WIP = "1742807779"
id_hoja_WIP_actualizado = "1505020649"

In [3]:
df_operadores = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                              id_hoja = id_hoja_operadores,
                                              desc_hoja = 'operadores', 
                                              flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 27
   ID_OPERADOR           OPERADOR                 AREA  ID_DIA_DESCANSO  \
0            1     ABIZAID ARAGÓN    ARMADO Y TAPIZADO                1   
1            2      ANAYELI LÓPEZ              GENERAL                6   
2            3  AQUILEO GUTIÉRREZ  ESTRUCTURA Y DISEÑO                1   

  DIA_DESCANSO FECHA_ACTUALIZACION_DESCANSO FECHA_INGRESO ESTATUS  
0        Lunes                   2026-04-19    1900-01-01  ACTIVO  
1       Sábado                   2026-04-19    1900-01-01  ACTIVO  
2        Lunes                   2026-04-19    1900-01-01  ACTIVO  
💾 Archivo guardado para lectura offline en: offline_databases\informacion_operadores_2026-04-25_06-14-35.csv


In [4]:
df_lista_actividades = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                              id_hoja = id_hoja_lista_actividades_actualizado,
                                              desc_hoja = 'lista_actividades', 
                                              flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 87
   ID_PROCESO  ID_SUBPROCESO KEY_SUBPROCESO            DESC_PROCESO  \
0           1              1            1.1  PROCESAMIENTO DE FIBRA   
1           1              2            1.2  PROCESAMIENTO DE FIBRA   
2           1              3          1.3.1  PROCESAMIENTO DE FIBRA   

                                       ACTIVIDAD    AREA  TIEMPO DEPENDENCIA  
0        Preparación de moldes (Limpieza y cera)  FIBRAS     1.0              
1                          Aplicación de Gelcoat  FIBRAS     0.5         1.1  
2  Laminado de capas de fibra de vidrio (Frente)  FIBRAS     0.5         1.2  
💾 Archivo guardado para lectura offline en: offline_databases\informacion_lista_actividades_2026-04-25_06-14-38.csv


In [5]:
df_areas = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                        id_hoja = id_hoja_areas,
                                        desc_hoja = 'areas', 
                                        flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 15
   ID_AREA                 AREA  ID_SUBAREA     SUBAREA FECHA_CREACION ESTATUS
0        0               FIBRAS           1      Fibras     1900-01-01  ACTIVO
1        1  ESTRUCTURA Y DISEÑO           1      Diseño     1900-01-01  ACTIVO
2        1  ESTRUCTURA Y DISEÑO           2  Estructura     1900-01-01  ACTIVO
💾 Archivo guardado para lectura offline en: offline_databases\informacion_areas_2026-04-25_06-14-39.csv


In [6]:
df_wip = mis_funciones.lectura_hoja_gs(id_libro = id_libro,
                                        id_hoja = id_hoja_WIP_actualizado,
                                        desc_hoja = 'wip', 
                                        flag_lectura_offline = False)

🔄 Iniciando lectura online...
   - Total de registros: 435
  ID_UNIDAD  ID_PROCESO  ID_SUBPROCESO KEY_SUBPROCESO            DESC_PROCESO  \
0   CG4-188           1              1            1.1  PROCESAMIENTO DE FIBRA   
1   CG4-188           1              2            1.2  PROCESAMIENTO DE FIBRA   
2   CG4-188           1              3          1.3.1  PROCESAMIENTO DE FIBRA   

                                       ACTIVIDAD    AREA  TIEMPO DEPENDENCIA  \
0        Preparación de moldes (Limpieza y cera)  FIBRAS     1.0               
1                          Aplicación de Gelcoat  FIBRAS     0.5         1.1   
2  Laminado de capas de fibra de vidrio (Frente)  FIBRAS     0.5         1.2   

   FLAG_TERMINADO FECHA_ACTUALIZACION  
0               1           22/4/2026  
1               1           22/4/2026  
2               1           22/4/2026  
💾 Archivo guardado para lectura offline en: offline_databases\informacion_wip_2026-04-25_06-14-40.csv


In [7]:
# ==========================================
# 2. CONFIGURACIÓN PARA EL DÍA ACTUAL
# ==========================================
dia_de_asignacion = "Sábado"
fecha_de_asignacion = "2026-04-25"

df_ops_disponibles = df_operadores[df_operadores['DIA_DESCANSO'] != dia_de_asignacion].copy()
df_ops_disponibles['HORAS_ASIGNADAS'] = 0.0

# Capacidad estándar de jornada laboral
max_horas_turno = 8.0

In [8]:
# ==========================================
# 3. PREPARACIÓN DEL WORK IN PROGRESS (WIP)
# ==========================================
# Trabajamos sobre una copia para no alterar el DataFrame original
wip_sim = df_wip.copy()

# Limpieza de dependencias (convertir texto "1.1, 1.2" a listas de Python)
wip_sim['DEPENDENCIA'] = wip_sim['DEPENDENCIA'].fillna('').astype(str)
wip_sim['DEPS_LIST'] = wip_sim['DEPENDENCIA'].apply(lambda x: [d.strip() for d in x.split(',')] if x else [])

In [9]:
# ==========================================
# 4. FUNCIÓN DE VERIFICACIÓN DE PREVALENCIA
# ==========================================
def verificar_pre_requisitos(df, id_unidad, lista_deps):
    if not lista_deps:
        return True 
    
    for dep in lista_deps:
        # Buscamos si la dependencia ya está marcada como terminada (1) para esa unidad
        estado = df[(df['ID_UNIDAD'] == id_unidad) & (df['KEY_SUBPROCESO'] == dep)]
        if not estado.empty and estado['FLAG_TERMINADO'].iloc[0] == 0:
            return False 
    return True

In [13]:
asignaciones_realizadas = []
hubo_movimiento = True
iteracion = 0

# Definimos el límite estricto de horas por día para un operador
max_horas_turno = 8 

print(f"Iniciando asignación para el día: {dia_de_asignacion}...")

while hubo_movimiento and iteracion < 100:
    hubo_movimiento = False
    iteracion += 1
    
    # Solo miramos lo que falta por hacer
    pendientes = wip_sim[wip_sim['FLAG_TERMINADO'] == 0]
    
    for idx, tarea in pendientes.iterrows():
        # 1. ¿Están listas las tareas anteriores?
        if verificar_pre_requisitos(wip_sim, tarea['ID_UNIDAD'], tarea['DEPS_LIST']):
            
            # 2. ¿Hay alguien del área con tiempo disponible HOY?
            candidatos = df_ops_disponibles[
                (df_ops_disponibles['AREA'] == tarea['AREA']) & 
                (df_ops_disponibles['HORAS_ASIGNADAS'] + tarea['TIEMPO'] <= max_horas_turno)
            ]
            
            if not candidatos.empty:
                # 3. Asignar al que tenga menos carga (Balanceo)
                op = candidatos.sort_values('HORAS_ASIGNADAS').iloc[0]
                
                # --- CÁLCULO DE HORARIOS ---
                horas_previas = op['HORAS_ASIGNADAS']
                nueva_carga = horas_previas + tarea['TIEMPO']
                
                # Calculamos horario asumiendo inicio de turno a las 08:00
                hora_inicio = 8 + horas_previas
                hora_fin = 8 + nueva_carga
                
                # Formato visual HH:MM
                str_inicio = f"{int(hora_inicio):02d}:{int((hora_inicio%1)*60):02d}"
                str_fin = f"{int(hora_fin):02d}:{int((hora_fin%1)*60):02d}"
                
                # Registrar en la bitácora
                asignaciones_realizadas.append({
                    'FECHA_PROGRAMADA': fecha_de_asignacion,
                    'ID_UNIDAD': tarea['ID_UNIDAD'],
                    'ACTIVIDAD': tarea['ACTIVIDAD'],
                    'AREA': tarea['AREA'],
                    'TIEMPO': tarea['TIEMPO'],
                    'OPERADOR': op['OPERADOR'],
                    'CARGA_ACUMULADA_OP': nueva_carga,               # <--- Tu nuevo dato
                    'HORARIO_ESTIMADO': f"{str_inicio} - {str_fin}"  # <--- Bonus visual
                })
                
                # Actualizar carga del operador
                df_ops_disponibles.loc[df_ops_disponibles['ID_OPERADOR'] == op['ID_OPERADOR'], 'HORAS_ASIGNADAS'] = nueva_carga
                
                # Marcar como terminada virtualmente para liberar las siguientes tareas
                wip_sim.at[idx, 'FLAG_TERMINADO'] = 1
                
                hubo_movimiento = True
                
                # 🛑 ELIMINAMOS EL BREAK AQUÍ.
                # Al quitarlo, el for continúa con la SIGUIENTE unidad, garantizando 
                # que todas las unidades tengan la misma oportunidad de recibir operadores.

Iniciando asignación para el día: Sábado...


In [14]:
# ==========================================
# 6. RESULTADOS FINALES
# ==========================================
df_resultado_final = pd.DataFrame(asignaciones_realizadas)
df_resultado_final

""


In [ ]:
print("\n--- RESUMEN DEL DÍA 25 DE ABRIL ---")
print(f"Tareas asignadas: {len(df_resultado_final)}")
print(f"Uso de capacidad total de operadores: {df_ops_disponibles['HORAS_ASIGNADAS'].sum():.2f} horas hombre.")

# Guardar resultado
df_resultado_final.to_csv(f'offline_databases/planificacion_diaria/Plan_de_asignacion_{fecha_de_asignacion}.csv', index = False)

# Mostrar los primeros resultados del plan
print("\nPrimeras 5 asignaciones del día:")
display(df_resultado_final.head())


--- RESUMEN DEL DÍA 25 DE ABRIL ---
Tareas asignadas: 0
Uso de capacidad total de operadores: 127.58 horas hombre.

Primeras 5 asignaciones del día:


""


In [ ]:
# ==========================================
# 5. MOTOR DE ASIGNACIÓN (LÓGICA DE ANTICIPACIÓN)
# ==========================================
asignaciones_realizadas = []
hubo_movimiento = True
iteracion = 0

print(f"Iniciando asignación para el día: {dia_de_asignacion} {fecha_de_asignacion}...")

while hubo_movimiento and iteracion < 100:
    hubo_movimiento = False
    iteracion += 1
    
    # Solo miramos lo que falta por hacer
    pendientes = wip_sim[wip_sim['FLAG_TERMINADO'] == 0]
    
    for idx, tarea in pendientes.iterrows():
        # 1. ¿Están listas las tareas anteriores?
        if verificar_pre_requisitos(wip_sim, tarea['ID_UNIDAD'], tarea['DEPS_LIST']):
            
            # 2. ¿Hay alguien del área con tiempo disponible?
            candidatos = df_ops_disponibles[
                (df_ops_disponibles['AREA'] == tarea['AREA']) & 
                (df_ops_disponibles['HORAS_ASIGNADAS'] + tarea['TIEMPO'] <= max_horas_turno)
            ]
            
            if not candidatos.empty:
                # 3. Asignar al que tenga menos carga (Balanceo)
                op = candidatos.sort_values('HORAS_ASIGNADAS').iloc[0]
                
                # Registrar en la bitácora
                asignaciones_realizadas.append({
                    'FECHA_PROGRAMADA': '2026-04-25',
                    'ID_UNIDAD': tarea['ID_UNIDAD'],
                    'ACTIVIDAD': tarea['ACTIVIDAD'],
                    'AREA': tarea['AREA'],
                    'TIEMPO': tarea['TIEMPO'],
                    'OPERADOR': op['OPERADOR']
                })
                
                # Actualizar carga del operador
                df_ops_disponibles.loc[df_ops_disponibles['ID_OPERADOR'] == op['ID_OPERADOR'], 'HORAS_ASIGNADAS'] += tarea['TIEMPO']
                
                # Marcar como terminada virtualmente para liberar las siguientes tareas de la unidad
                wip_sim.at[idx, 'FLAG_TERMINADO'] = 1
                
                hubo_movimiento = True
                break # Reiniciar búsqueda para recalcular prioridades

## Parte 2

In [75]:
def obtener_dia_actual(fecha_simulacion):
    # Traducir nombre del día para el filtro de operadores
    dias = {0: "Lunes", 1: "Mares", 2: "Miércoles", 3: "Jueves", 4: "Viernes", 5: "Sábado", 6: "Domingo"}
    return dias[fecha_simulacion.weekday()]

In [76]:
df_lista_actividades['TIEMPO'] = pd.to_numeric(df_lista_actividades['TIEMPO'], errors='coerce').fillna(0)

# Función segura para convertir llaves 1_1 a float 1.1
def key_to_val(k):
    if pd.isna(k) or str(k).strip() == '': 
        return 0.0
    return float(str(k).replace('_', '.'))

In [77]:
fecha_hoy_str = "20/04/2026"
fecha_hoy = datetime.strptime(fecha_hoy_str, "%d/%m/%Y")
dia_nombre = obtener_dia_actual(fecha_hoy)
    
if dia_nombre == "Domingo":
    print("Hoy es domingo. La planta está cerrada. ¡Vaya a descansar!")

model = cp_model.CpModel()
HORIZONTE = 60*8

In [78]:
# 1. FILTRAR OPERADORES DISPONIBLES
# Solo los que no descansan hoy y están activos
operadores_disponibles = df_operadores[(df_operadores['DIA_DESCANSO'] != dia_nombre) & 
                            (df_operadores['ESTATUS'] == 'ACTIVO')].to_dict('records')

display(df_operadores[df_operadores['DIA_DESCANSO'] != dia_nombre]['AREA'].value_counts())
display(df_operadores.head(5))

AREA
ESTRUCTURA Y DISEÑO    6
DETALLADO              4
LAMINADO               3
ARMADO Y TAPIZADO      3
PINTURA                2
ELÉCTRICO              2
FIBRAS                 2
GENERAL                1
Name: count, dtype: int64

,ID_OPERADOR,OPERADOR,AREA,ID_DIA_DESCANSO,DIA_DESCANSO,FECHA_ACTUALIZACION_DESCANSO,FECHA_INGRESO,ESTATUS
0,1,ABIZAID ARAGÓN,ARMADO Y TAPIZADO,1,Lunes,2026-04-19,1900-01-01,ACTIVO
1,2,ANAYELI LÓPEZ,GENERAL,6,Sábado,2026-04-19,1900-01-01,ACTIVO
2,3,AQUILEO GUTIÉRREZ,ESTRUCTURA Y DISEÑO,1,Lunes,2026-04-19,1900-01-01,ACTIVO
3,4,DAVID MOLINA,LAMINADO,2,Martes,2026-04-19,1900-01-01,ACTIVO
4,5,DAVID SALDÍVAR,PINTURA,2,Martes,2026-04-19,1900-01-01,ACTIVO


In [79]:
# 2. PROCESAMIENTO DE TAREAS Y UNIDADES
# --- INICIALIZACIÓN ---
task_vars = {}
lista_para_consolidar = [] # Lista temporal para guardar los hallazgos por bus

# Función robusta para comparar "1_1", "1.1", "1_10", etc.
def key_to_tuple(k):
    try:
        k_str = str(k).replace('_', '.') # Normalizamos a punto para el split
        if pd.isna(k) or k_str.strip() in ["", "nan", "-"]:
            return (0, 0)
        return tuple(map(int, k_str.split('.')))
    except:
        return (0, 0)

# --- PROCESAMIENTO ---
for _, bus in df_wip.iterrows():
    u_id = bus['ID_UNIDAD']
    
    # Manejo seguro de nulos en el ID actual
    val_raw = bus['KEY_SUBPROCESO_MAS_RECIENTE']
    ultimo_id = str(val_raw).strip() if pd.notna(val_raw) else "0_0"
    
    # 1. Prioridad por días en planta
    try:
        f_ingreso = datetime.strptime(str(bus['FECHA_INGRESO_PRODUCCION']).strip(), "%d/%m/%Y")
        dias_en_planta = (fecha_hoy - f_ingreso).days
    except:
        dias_en_planta = 0
    
    peso_prioridad = dias_en_planta * 100 

    # 2. Filtrado de Actividades
    val_ultimo = key_to_tuple(ultimo_id)
    
    # Aplicamos el filtro comparando tuplas
    mask = df_lista_actividades['KEY_SUBPROCESO'].apply(key_to_tuple) > val_ultimo
    proximas_actividades = df_lista_actividades[mask].copy()

    if not proximas_actividades.empty:
        # Ordenamos correctamente (1_9 antes que 1_10)
        proximas_actividades['sort_key'] = proximas_actividades['KEY_SUBPROCESO'].apply(key_to_tuple)
        proximas_actividades = proximas_actividades.sort_values('sort_key').head(3)
        
        # AGREGAR AL CONSOLIDADO:
        # Creamos una copia para el reporte, agregando el ID del bus y la prioridad
        temp_reporte = proximas_actividades.copy()
        temp_reporte.insert(0, 'ID_UNIDAD', u_id)
        temp_reporte['PRIORIDAD_PUNTOS'] = peso_prioridad
        lista_para_consolidar.append(temp_reporte)

        # 3. Asignación de variables al modelo (tu lógica original)
        for _, tarea in proximas_actividades.iterrows():
            a_key = tarea['KEY_SUBPROCESO']
            t_raw = str(tarea['TIEMPO']).strip()
            duracion_min = int(float(t_raw) * 60) if t_raw not in ['', 'nan', 'None'] else 0
            
            if duracion_min == 0: continue 

            area_tarea = str(tarea['AREA']).strip()
            operadores_aptos = [
                o['ID_OPERADOR'] for o in operadores_disponibles 
                if str(o['AREA']).strip() == area_tarea
            ]
            
            if not operadores_aptos:
                continue

            suffix = f"{u_id}_{a_key}".replace(".", "_")
            start = model.NewIntVar(0, HORIZONTE, f'start_{suffix}')
            end = model.NewIntVar(0, HORIZONTE, f'end_{suffix}')
            interval = model.NewIntervalVar(start, duracion_min, end, f'inter_{suffix}')
            op_var = model.NewIntVarFromDomain(cp_model.Domain.FromValues(operadores_aptos), f'op_{suffix}')
            
            task_vars[(u_id, a_key)] = {
                'start': start, 'end': end, 'interval': interval, 
                'op': op_var, 'peso': peso_prioridad, 'dur': duracion_min,
                'dep': tarea['DEPENDENCIA']
            }

# --- CREACIÓN DEL DATAFRAME FINAL ---
if lista_para_consolidar:
    consolidaciones_proximas_actividades = pd.concat(lista_para_consolidar, ignore_index=True)
    # Limpiamos columnas auxiliares
    if 'sort_key' in consolidaciones_proximas_actividades.columns:
        del consolidaciones_proximas_actividades['sort_key']
    
    print("Consolidación completada con éxito.")
    display(consolidaciones_proximas_actividades.head(10))
else:
    print("No se encontraron actividades próximas para ninguna unidad.")

Consolidación completada con éxito.


,ID_UNIDAD,ID_PROCESO,ID_SUBPROCESO,KEY_SUBPROCESO,DESC_MACROPROCESO,ACTIVIDAD,AREA,TIEMPO,DEPENDENCIA,PRIORIDAD_PUNTOS
0,CG4-188,9,3,9.3,DETALLADO Y SALIDA,Instalar calcas exteriores y biseles,DETALLADO,1.5,8.8,2400
1,CG4-188,9,4,9.4,DETALLADO Y SALIDA,Sellar estribos y aplicar capa de goma,DETALLADO,2.0,9.1,2400
2,CG4-188,9,5,9.5,DETALLADO Y SALIDA,Instalación de vidrios y parabrisas,DETALLADO,3.0,9.4,2400
3,CG4-190,8,2,8.2,ARMADO E INTERIORES,Adaptar y soldar ductos de puertas,ARMADO Y TAPIZADO,2.0,8.1,1700
4,CG4-190,8,3,8.3,ARMADO E INTERIORES,Instalar y pijar ductería general,ARMADO Y TAPIZADO,3.0,8.2,1700
5,CG4-190,8,4,8.4,ARMADO E INTERIORES,Instalar falleba y aro de falleba,ARMADO Y TAPIZADO,1.0,8.3,1700
6,CG4-191,7,2,7.2,PREPARACIÓN Y PINTURA,Aplicar fondo plaste en carrocería,PINTURA,3.0,7.1,1300
7,CG4-191,7,3,7.3,PREPARACIÓN Y PINTURA,Aplicar fondo epóxico en techo,PINTURA,1.5,7.1,1300
8,CG4-191,7,4,7.4,PREPARACIÓN Y PINTURA,Empapelar unidad completa,PINTURA,2.0,7.2,1300
9,CG4-193,4,2,4.2,LAMINADO Y CNC,Fabricar charola para puerta de baterías,LAMINADO,0.5,,600


In [80]:
consolidaciones_proximas_actividades.groupby(["ID_UNIDAD", "AREA"])['TIEMPO'].sum()

ID_UNIDAD  AREA               
CG4-188    DETALLADO              6.5
CG4-190    ARMADO Y TAPIZADO      6.0
CG4-191    PINTURA                6.5
CG4-193    LAMINADO               2.0
CG4-194    ESTRUCTURA Y DISEÑO    3.0
CG4-195    ESTRUCTURA Y DISEÑO    4.0
           FIBRAS                 2.0
CG4-196    FIBRAS                 4.0
Name: TIEMPO, dtype: float64

In [81]:
# 3. RESTRICCIONES TÉCNICAS
    
# A. Flujo Serial: La tarea B solo empieza si la A terminó hoy
for (u_id, a_key), task in task_vars.items():
    dep_key = task['dep']
    if (u_id, dep_key) in task_vars:
        model.Add(task['start'] >= task_vars[(u_id, dep_key)]['end'])

# B. Exclusividad del Operador: No-Overlap (Cero duplicidad de tareas)
for op in operadores_disponibles:
    ops_intervals = []
    for (u_id, a_key), task in task_vars.items():
        # Variable booleana para verificar si el operador o es el elegido
        is_assigned = model.NewBoolVar(f'bool_{op["ID_OPERADOR"]}_{u_id}_{a_key}')
        model.Add(task['op'] == op['ID_OPERADOR']).OnlyEnforceIf(is_assigned)
        model.Add(task['op'] != op['ID_OPERADOR']).OnlyEnforceIf(is_assigned.Not())
            
        # Intervalo opcional
        opt_inter = model.NewOptionalIntervalVar(task['start'], task['dur'], task['end'], 
                                                    is_assigned, f'opt_{op["ID_OPERADOR"]}_{u_id}_{a_key}')
        ops_intervals.append(opt_inter)
        
    if ops_intervals:
        model.AddNoOverlap(ops_intervals)

In [82]:
# 4. OPTIMIZACIÓN: Maximizar avance en unidades críticas
if not task_vars:
    print("ERROR: No se generaron tareas. Verifique que las unidades tengan actividades pendientes en la secuencia.")
else:
    print(f"Intentando programar {len(task_vars)} tareas con {len(operadores_disponibles)} operadores.")

# Penalizamos el tiempo de finalización (queremos que terminen lo antes posible)
# Multiplicado por el peso de prioridad (días en planta)
model.Minimize(sum(task['end'] * task['peso'] for task in task_vars.values()))

# 5. RESULTADOS
solver = cp_model.CpSolver()
status = solver.Solve(model)

Intentando programar 21 tareas con 23 operadores.


In [83]:
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    data_final = []
    for (u_id, a_key), task in task_vars.items():
        if solver.Value(task['end']) > 0:
            op_id = solver.Value(task['op'])
            nombre_op = next(o['OPERADOR'] for o in operadores_disponibles if o['ID_OPERADOR'] == op_id)
            data_final.append({
                'Operador': nombre_op, 'Unidad': u_id, 'Actividad': a_key,
                'Inicio (Min)': solver.Value(task['start']), 'Fin (Min)': solver.Value(task['end'])
            })
            df_asignaciones = pd.DataFrame(data_final).sort_values(['Operador', 'Inicio (Min)'])
    
    print("No se pudo generar un plan de trabajo. Verifique disponibilidad de áreas.")

No se pudo generar un plan de trabajo. Verifique disponibilidad de áreas.


In [84]:
if status == cp_model.INFEASIBLE:
    print("EL MODELO ES INFACTIBLE. Causas probables:")
    # Revisar si hay alguna tarea sin operadores aptos
    for (u, a), info in task_vars.items():
        # Esto te dirá qué tarea específica está causando el bloqueo
        pass 
    print("- Verifique que no haya tareas que duren más de 9 horas.")
    print("- Verifique que haya al menos un operador disponible para cada área activa hoy.")